# Multilayer Perceptron MLP Model

In [1]:
import pandas as pd # type: ignore
Data_final = pd.read_csv('/Users/instructorzamora/Documents/3_Maestria_Estadistica_UNINORTE/3_Tercer_Semestre/Machine_Learning/Deteccion_Fraude/Data_final.csv')
Data_final

,D12,D14,D11,D8,TransAmt,D3,D7,dist1,dist2,V209,...,V285,id_01,D13,isFraud,card4_discover,card4_mastercard,card4_visa,card6_credit,card6_debit,card6_debit or credit
0,0.0,0.0,13.0,37.875,68.500000,13.0,0.0,19.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,1,0,0,1,0,0
1,0.0,0.0,43.0,37.875,29.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,1,0,0
2,0.0,0.0,315.0,37.875,59.000000,8.0,0.0,287.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,0,1,0,1,0
3,0.0,0.0,43.0,37.875,50.000000,0.0,0.0,8.0,37.0,0.0,...,10.0,-5.0,0.0,0.0,0,1,0,0,1,0
4,0.0,0.0,43.0,37.875,50.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,0.0,0.0,0.0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,0.0,0.0,56.0,37.875,49.000000,30.0,0.0,48.0,37.0,0.0,...,1.0,-5.0,0.0,0.0,0,0,1,0,1,0
590536,0.0,0.0,0.0,37.875,39.500000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590537,0.0,0.0,0.0,37.875,30.950001,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590538,0.0,0.0,22.0,37.875,117.000000,0.0,0.0,3.0,37.0,0.0,...,5.0,-5.0,0.0,0.0,0,1,0,0,1,0


El dataset final es un conjunto de datos extenso de detección de fraude con 590,540 registros y 22 columnas, diseñado para un modelo de machine learning que busca identificar transacciones fraudulentas. Contiene variables numéricas como 'TransAmt' (monto de transacción), 'dist1', 'dist2', y códigos como D12, D14, D11, junto con variables categóricas binarias que representan características de tarjetas de pago (como tipos de tarjetas Discover, Mastercard, Visa, y tipos de tarjetas de crédito/débito). La variable objetivo 'isFraud' es binaria (0 o 1), indicando si una transacción es fraudulenta, mientras que la mayoría de las otras variables son numéricas con muchos valores cercanos a cero, sugiriendo un preprocesamiento de datos previo. Este dataset parece estar preparado para entrenar un modelo de clasificación que pueda predecir la probabilidad de fraude en transacciones financieras.

## Metricas Multilayer Perceptron

In [17]:
# ------------------------
# Paso 1: Importar paquetes necesarios
# ------------------------
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from joblib import dump
from time import time
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from scipy.stats import uniform, loguniform

# ------------------------
# Paso 2: Cargar y preparar los datos
# ------------------------
y_train = Data_final['isFraud']
x_train = Data_final.drop(columns=['isFraud']).select_dtypes(include='number')  # Solo numéricas
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.20,random_state=80,stratify=y)

# ------------------------
# Paso 3: Pipeline con SMOTE + Escalado + MLP
# ------------------------
pipe_mlp = Pipeline([
    ('smote', SMOTE(random_state=42, sampling_strategy='auto')),  # Ajustado para rendimiento
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=100, solver='adam', early_stopping=True,  # Cambiado a adam y early_stopping
                        validation_fraction=0.1, n_iter_no_change=5, 
                        random_state=42))
])

# ------------------------
# Paso 4: Espacio de búsqueda usando RandomizedSearchCV (más rápido que Bayes)
# ------------------------
param_dist = {
    'mlp__hidden_layer_sizes': [(50,), (100,)],
    'mlp__alpha': loguniform(1e-4, 1e-2),
    'mlp__learning_rate_init': loguniform(1e-4, 1e-2),
    'mlp__activation': ['relu']
}

# ------------------------
# Paso 5: RandomizedSearchCV (más eficiente que BayesSearchCV)
# ------------------------
random_search = RandomizedSearchCV(
    estimator=pipe_mlp,
    param_distributions=param_dist,
    n_iter=6,          # Reducido de 10 a 6
    scoring='roc_auc',
    cv=2,              # Reducido de 3 a 2
    n_jobs=-1,
    random_state=42,
    verbose=1          # Agregado para ver progreso
)

# Medir tiempo de entrenamiento
print("Iniciando entrenamiento...")
start_time = time()
random_search.fit(x_train, y_train)
training_time_mlp = time() - start_time
print(f"Entrenamiento completado en {training_time_mlp:.2f} segundos")

# Guardar el modelo
dump(random_search, 'mlp_smote_optimized_fast_m1.joblib')

# ------------------------
# Paso 6: Predicciones
# ------------------------
print("Realizando predicciones...")
x_test_num = x_test.select_dtypes(include='number')
y_pred_mlp = random_search.best_estimator_.predict(x_test_num)
y_pred_proba_mlp = random_search.best_estimator_.predict_proba(x_test_num)[:, 1]

# ------------------------
# Paso 7: Métricas
# ------------------------
precision_mlp = precision_score(y_test, y_pred_mlp, average='weighted')
recall_mlp = recall_score(y_test, y_pred_mlp, average='weighted')
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)
f1_mlp = f1_score(y_test, y_pred_mlp, average='weighted')
auc_mlp = roc_auc_score(y_test, y_pred_proba_mlp)

# ------------------------
# Paso 8: Resultados
# ------------------------
resultados_mlp = pd.DataFrame({
    'Precision': [f"{precision_mlp:.2f}"],
    'Recall': [f"{recall_mlp:.2f}"],
    'Accuracy': [f"{accuracy_mlp:.2f}"],
    'F1-Score': [f"{f1_mlp:.2f}"],
    'AUC': [f"{auc_mlp:.2f}"],
    'CPU time (s)': [round(training_time_mlp, 2)]
})
print("Métricas MLP + SMOTE optimizado y rápido para M1:")
display(resultados_mlp)

# Mostrar los mejores parámetros
print("Mejores parámetros encontrados:")
print(random_search.best_params_)

Iniciando entrenamiento...
Fitting 2 folds for each of 6 candidates, totalling 12 fits
Entrenamiento completado en 9588.59 segundos
Realizando predicciones...
Métricas MLP + SMOTE optimizado y rápido para M1:


,Precision,Recall,Accuracy,F1-Score,AUC,CPU time (s)
0,0.95,0.88,0.88,0.91,0.79,9588.59


Mejores parámetros encontrados:
{'mlp__activation': 'relu', 'mlp__alpha': 0.0005611516415334506, 'mlp__hidden_layer_sizes': (50,), 'mlp__learning_rate_init': 0.00023273922280628724}


Para el modelo de Perceptrón Multicapa (MLP), el análisis muestra un rendimiento sólido con algunas particularidades. La precisión del 95% indica una alta capacidad para identificar correctamente las transacciones fraudulentas. Sin embargo, el recall del 88% sugiere que el modelo podría estar perdiendo algunas transacciones fraudulentas reales. La accuracy del 88% refleja un rendimiento general bueno, aunque ligeramente inferior a los modelos anteriores. El F1-Score de 0.91 representa un equilibrio razonable entre precisión y recall, compensando la diferencia entre ambas métricas. El AUC de 0.79 es prometedor, indicando una buena capacidad para discriminar entre clases. El tiempo de CPU de 9,588.59 segundos (aproximadamente 2.7 horas) es significativamente menor comparado con los modelos KNN y Regresión Logística, lo que podría sugerir una mayor eficiencia computacional del MLP en este conjunto de datos específico.